# Fleet Operations Analytics
**Damarius McNair** - [github.com/DCodeBase-X](https://github.com/DCodeBase-X)

This notebook walks through the core analytical approach used to identify overtime cost drivers,
track fleet utilization efficiency, and support capacity planning decisions across a 5,200+ unit
regional fleet; the same methodology that delivered a **23% reduction in overtime costs** and
a **15% improvement in staffing efficiency**.


### Contents
1. _Data overview & quality check_
2. _Fleet utilization analysis - [including seasonal demand by vehicle type & location_]
3. _Overtime cost root-cause analysis_
4. _Maintenance impact on capacity_
5. _Key findings & recommendations_

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

# Ensure output directory exists
os.makedirs('../visuals', exist_ok=True)

# Load data
util  = pd.read_csv('../data/daily_utilization.csv',  parse_dates=['date'])
ot    = pd.read_csv('../data/staff_overtime.csv',      parse_dates=['date'])
maint = pd.read_csv('../data/maintenance_records.csv', parse_dates=['date'])
veh   = pd.read_csv('../data/fleet_vehicles.csv',      parse_dates=['acquired_date'])

print(f'Utilization records : {len(util):,}')
print(f'Overtime records    : {len(ot):,}')
print(f'Maintenance records : {len(maint):,}')
print(f'Fleet vehicles      : {len(veh):,}')

## 1. Data Overview

In [ ]:
# Date range and basic summary
print('Date range:', util['date'].min().date(), '→', util['date'].max().date())
print('\nUtilization summary:')
display(util[['utilization_rate','hours_used','miles_driven']].describe().round(2))

## 2. Fleet Utilization Analysis

In [ ]:
# Monthly average utilization
monthly = util.groupby(util['date'].dt.to_period('M'))['utilization_rate'].mean() * 100

fig, ax = plt.subplots()
monthly.plot(ax=ax, marker='o', linewidth=2, color='#4F46E5')
ax.axhline(80, color='#10B981', linestyle='--', label='Target 80%')
ax.set_title('Monthly Fleet Utilization (%)')
ax.set_ylabel('Utilization %')
ax.set_xlabel('')
ax.legend()
plt.tight_layout()
plt.savefig('../visuals/utilization_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# Utilization by location × vehicle type
pivot = util.groupby(['location','vehicle_type'])['utilization_rate'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Utilization %'})
ax.set_title('Fleet Utilization (%) — Location × Vehicle Type')
plt.tight_layout()
plt.savefig('../visuals/utilization_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Seasonal utilization by vehicle type
# Compact/Mid-Size peak in summer (vacations); SUV/Truck peak in winter (tournaments, cold weather)
monthly_type = (
    util.assign(month=util['date'].dt.month)
    .groupby(['month', 'vehicle_type'])['utilization_rate']
    .mean().unstack() * 100
)

fig, ax = plt.subplots(figsize=(13, 5))
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for vtype, color in zip(monthly_type.columns,
                        ['#4F46E5','#818CF8','#10B981','#F59E0B','#EF4444']):
    ax.plot(range(1, 13), monthly_type[vtype], marker='o', label=vtype,
            color=color, linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_title('Seasonal Utilization by Vehicle Type (%)')
ax.set_ylabel('Utilization %')
ax.axvspan(6, 8, alpha=0.08, color='orange', label='Summer peak')
ax.axvspan(11, 12, alpha=0.08, color='steelblue', label='Winter events')
ax.legend(loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.22))
plt.tight_layout()
plt.savefig('../visuals/seasonal_by_type.png', bbox_inches='tight')
plt.show()

In [ ]:
# Seasonal utilization by location
# South/West lead in summer (warm climate, tourism); North/Central dip in winter
monthly_loc = (
    util.assign(month=util['date'].dt.month)
    .groupby(['month', 'location'])['utilization_rate']
    .mean().unstack() * 100
)

fig, ax = plt.subplots(figsize=(13, 5))
loc_colors = {'South': '#EF4444', 'West': '#F59E0B',
              'East': '#10B981', 'Central': '#818CF8', 'North': '#4F46E5'}
for loc in monthly_loc.columns:
    ax.plot(range(1, 13), monthly_loc[loc], marker='o', label=loc,
            color=loc_colors[loc], linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_title('Seasonal Utilization by Location (%)')
ax.set_ylabel('Utilization %')
ax.axvspan(6, 8, alpha=0.08, color='orange', label='Summer peak')
ax.axvspan(11, 12, alpha=0.08, color='steelblue', label='Winter events')
ax.legend(loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.22))
plt.tight_layout()
plt.savefig('../visuals/seasonal_by_location.png', bbox_inches='tight')
plt.show()

## 3. Overtime Cost Root-Cause Analysis

In [ ]:
OT_PREMIUM = 28.0  # $/hr

# OT cost by location
ot_loc = ot.groupby('location')['overtime_hours'].sum().sort_values(ascending=False)
ot_cost_loc = ot_loc * OT_PREMIUM

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ot_cost_loc.plot(kind='bar', ax=axes[0], color='#818CF8')
axes[0].set_title('OT Cost by Location ($)')
axes[0].set_ylabel('Cost ($)')
axes[0].tick_params(axis='x', rotation=30)

ot_role = ot.groupby('role')['overtime_hours'].sum().sort_values(ascending=False)
(ot_role * OT_PREMIUM).plot(kind='bar', ax=axes[1], color='#4F46E5')
axes[1].set_title('OT Cost by Role ($)')
axes[1].set_ylabel('Cost ($)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../visuals/overtime_breakdown.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly OT trend
monthly_ot = ot.groupby(ot['date'].dt.to_period('M'))['overtime_hours'].sum()

fig, ax = plt.subplots()
monthly_ot.plot(kind='bar', ax=ax, color='#C7D2FE', edgecolor='#4F46E5')
ax.set_title('Monthly Overtime Hours')
ax.set_ylabel('Overtime Hours')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../visuals/monthly_overtime.png', bbox_inches='tight')
plt.show()

print(f'Peak OT month : {monthly_ot.idxmax()} ({monthly_ot.max():,.0f} hrs)')
print(f'Low  OT month : {monthly_ot.idxmin()} ({monthly_ot.min():,.0f} hrs)')

## 5. Key Findings

| Finding | Detail |
|---|---|
| **Overtime peak** | Summer months (June–August) account for ~40% of annual OT spend driven by staff vacations and peak demand; weekends included in Jun–Aug shifts |
| **Top OT driver** | Service Agents and Lot Attendants generate ~60% of total OT cost |
| **Seasonal vehicle demand** | Compact & Mid-Size utilization spikes Jun–Aug (vacation road trips); SUV/Truck demand rises Nov–Feb (tournaments, cold-weather events, etc.) |
| **High-demand locations (summer)** | South & West locations lead utilization Jun–Aug — warm climate and tourism drives rentals |
| **Low-demand locations (winter)** | North and Central drop significantly Jan–Feb; recover partially in Nov–Dec due to holiday event travel |
| **Under-utilized assets** | ~8% of fleet consistently below 50% utilization — reallocation candidates |
| **Maintenance hotspot** | Engine repairs and brake service peak in winter (cold-weather stress); oil changes and tire rotations peak in summer (high mileage) |

### Recommended Actions
1. **Summer staffing plan** — add contract coverage June–August at South and West locations to reduce OT premium exposure during staff vacation season

2. **Seasonal fleet rebalancing** — shift Compact/Mid-Size inventory toward South/West for summer; rotate SUVs/Trucks toward North/Central for winter tournament season

3. **Stagger shift scheduling** at high-OT locations to flatten the Monday/Friday spike and cover weekend demand in peak months

4. **Pre-winter maintenance push** — schedule engine and brake inspections in October before cold-weather failure rates climb

5. **Predictive maintenance** — flag vehicles approaching engine repair thresholds early, especially in North/Central ahead of winter

In [ ]:
# Maintenance cost by type
maint_type = maint.groupby('maintenance_type')['cost'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
maint_type.plot(kind='barh', ax=ax, color='#A5B4FC')
ax.set_title('Maintenance Cost by Type ($)')
ax.set_xlabel('Total Cost ($)')
plt.tight_layout()
plt.savefig('../visuals/maintenance_cost.png', bbox_inches='tight')
plt.show()

## 5. Key Findings

| Finding | Detail |
|---|---|
| **Overtime peak** | Summer months (June–August) account for ~35% of annual OT spend |
| **Top OT driver** | Service Agents and Lot Attendants generate ~60% of total OT cost |
| **Under-utilized assets** | ~8% of fleet consistently below 50% utilization — reallocation candidates |
| **High-demand locations** | North and West locations exceed 85% utilization — capacity gap signals |
| **Maintenance hotspot** | Engine repairs drive 40%+ of total maintenance cost despite low frequency |

### Recommended Actions
1. **Stagger shift scheduling** at high-OT locations to flatten the Monday/Friday spike

2. **Reallocate low-utilization vehicles** from East to North/West to close capacity gap

3. **Predictive maintenance** — flag vehicles approaching engine repair thresholds early

4. **Summer surge staffing plan** — add contract coverage June–August to reduce OT premium exposure